In [17]:
import os, random

import pytorch_lightning as pl
from PIL import Image
from pytorch_lightning.callbacks import ModelCheckpoint
from torch import nn, argmax, randint
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torchvision import transforms
from torchvision.datasets import MNIST
from torchmetrics import Accuracy
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, random_split, TensorDataset


In [18]:
class DigitModel(pl.LightningModule):
    def __init__(self) -> None:
        super().__init__()

        self.training_step_outputs = []

        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

        self.test_accuracy = Accuracy(task="multiclass", num_classes=10)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=10)

    def forward(self, x):
        batch_size, channels, height, width = x.size()

        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        outputs = self(x)
        criterion = nn.CrossEntropyLoss()
        self.loss = criterion(outputs, y)

        self.log("train_loss", self.loss, prog_bar=True, on_step=True, on_epoch=True)
        return self.loss
    
    def validation_step(self, valid_batch, batch_idx): 
        x, y = valid_batch
        logits = self(x)
        loss = F.nll_loss(logits, y)
        preds = argmax(logits, dim=1)
        self.val_accuracy.update(preds, y)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.val_accuracy, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.nll_loss(y_hat, y)
        preds = argmax(y_hat, dim=1)
        self.test_accuracy.update(preds, y)
        
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", self.test_accuracy, prog_bar=True)
        

    # def on_test_epoch_end(self):
    #     correct_score = sum([x["correct"] for x in self.training_step_outputs])
    #     total = sum([x["total"] for x in self.training_step_outputs])
    #     acc = correct_score/total
    #     self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)


    # def on_train_epoch_start(self) -> None:
    #     print('\n')

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=1e-3)

In [19]:
class DigitDataModel(pl.LightningDataModule):
    def __init__(self) -> None:
        super().__init__()
        self.download_dir = ''
        self.batch_size = 64
        self.transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
        

    def setup(self, stage: str) -> None:
        
        if stage == 'fit' or stage is None:
            mnist_full = MNIST(self.download_dir, train=True, download=True, transform=self.transform)
            self.mnist_train, self.mnist_val = random_split(mnist_full, [55000, 5000])

        if stage == 'test' or stage is None:
            self.mnist_test = MNIST(self.download_dir, train=False, download=True, transform=self.transform)





    def train_dataloader(self):
        return DataLoader(self.mnist_train, batch_size=self.batch_size, shuffle=True)

    def test_dataloader(self):
        return DataLoader(self.mnist_test, batch_size=self.batch_size, shuffle=True)
    
    def val_dataloader(self):
        return DataLoader(self.mnist_val, batch_size=self.batch_size)



In [5]:
digit = DigitDataModel()
model = DigitModel()

checkpoint_callback = ModelCheckpoint(
        monitor='val_loss',
        dirpath='./checkpoints/',
        filename='mnist-epoch{epoch:02d}-val_loss{val/loss:.2f}',
        save_top_k=1,
        mode='min',
        auto_insert_metric_name=False
        )


trainer = pl.Trainer(max_epochs=5, devices=1, callbacks=[checkpoint_callback])
trainer.fit(model, digit)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
C:\Users\jerem\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:67: UserWarning: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
  warning_cache.warn(
C:\Users\jerem\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:612: UserWarning: Checkpoint directory C:\Users\jerem\Desktop\School\CSE4310 Computer Vision\Project Demo\checkpoints exists and is not empt

Sanity Checking: 0it [00:00, ?it/s]

C:\Users\jerem\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:430: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
C:\Users\jerem\AppData\Local\Temp\ipykernel_10076\4042139247.py:25: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.log_softmax(x)
C:\Users\jerem\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:430: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in t

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


In [6]:
trainer.test(model, digit)

C:\Users\jerem\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:478: PossibleUserWarning: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
  rank_zero_warn(
C:\Users\jerem\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:430: PossibleUserWarning: The dataloader, test_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Testing: 0it [00:00, ?it/s]

C:\Users\jerem\AppData\Local\Temp\ipykernel_10076\4042139247.py:25: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.log_softmax(x)


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9850999712944031     │
│         test_loss         │    0.04852030426263809    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.04852030426263809, 'test_acc': 0.9850999712944031}]

In [9]:
best_chkpt = checkpoint_callback.best_model_path
model.eval()
model.load_from_checkpoint(best_chkpt)
folder = "./Test Images/"
images = [f for f in os.listdir(folder)]

In [15]:
random_image = random.choice(images)
image = Image.open(os.path.join(folder, random_image))
image

In [11]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
image_tensor = transform(image).unsqueeze(0)

output = model(image_tensor)
prediction = output.argmax(dim=1).item()
prediction


C:\Users\jerem\AppData\Local\Temp\ipykernel_10076\4042139247.py:25: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.log_softmax(x)


3

In [16]:
"""
Code Report
NOTE: Current version ran under the CPU, but can be modified to the GPU with
a parameter (accelerator = "gpu") after the max_epoch argument in pytorch_lighting's
Trainer.
NOTE: Deprecated functions are used, but does not dampens the demo's functionality.

This is a program that teaches the computer how to recognize digits from images. The
dataset used for training & testing is the MNISIT dataset. However, there are test
images that are custom drawn inside of paint.

The code organize the transformation/model for the data inside of class and does
the same thing for the data allowing efficent organization of code for testing
and training. After an instance is made for these classes the computer then begins
training the data calculating training loss with Cross Entropy.

After this training is done, the computer is tested with various images. Common parses
of the program put the accuracy commonly over 90% with a negligible loss. After a
table is displayed with the accuracy and lost an image is displayed of the kind of data
in the data sets. Lastly, the images output and prediction is calculated.
"""

'\nCode Report\nNOTE: Current version ran under the CPU, but can be modified to the GPU with\na parameter (accelerator = "gpu") after the max_epoch argument in pytorch_lighting\'s\nTrainer.\nNOTE: Deprecated functions are used, but does not dampens the demo\'s functionality.\n\nThis is a program that teaches the computer how to recognize digits from images. The\ndataset used for training & testing is the MNISIT dataset. However, there are test\nimages that are custom drawn inside of paint.\n\nThe code organize the transformation/model for the data inside of class and does\nthe same thing for the data allowing efficent organization of code for testing\nand training. After an instance is made for these classes the computer then begins\ntraining the data calculating training loss with Cross Entropy.  \n'